# TV2 Tuần 1 — xử lý ảnh và tổng hợp 3 kênh
Chạy lần lượt các cell trên **Google Colab CPU**. Notebook đọc kết quả TV1 trong `data_real` trên Google Drive và ghi báo cáo, hình QC cùng bộ dữ liệu TV2 (nếu có VOI Pre/Post1 trung gian) trở lại Drive. Không chạy lại Step 1–2 và không train mô hình.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/CS406-SCB-TumorSeg')
TV2 = PROJECT / 'TV2_Tuan1'
DATA = PROJECT / 'data_real'
assert (TV2 / 'run_step3.py').is_file(), f'Chưa tìm thấy thư mục TV2: {TV2}'
assert (DATA / 'splits' / 'splits_metadata.csv').is_file(), f'Chưa tìm thấy metadata TV1: {DATA}'
print('TV2 code:', TV2)
print('TV1 data:', DATA)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(TV2 / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], cwd=TV2, check=True)


## Chọn chế độ
`auto`: nếu có đủ `data_real/processed/voi_step2/{CASE}.npz` chứa `pre_voi` và `post1_voi`, notebook sẽ tạo bộ 3 kênh mới tại `processed/npz_tv2_step3/`. Nếu không, notebook QC bộ 3 kênh TV1 đã tạo và ghi rõ giới hạn không thể tái tạo Step 3 độc lập. Chọn `validate` để chỉ QC, hoặc `recompute` để yêu cầu VOI trung gian bắt buộc. Dữ liệu TV1 không bị ghi đè.


In [ ]:
MODE = 'auto'  # 'auto', 'validate', hoặc 'recompute'
PILOT_SIZE = 12
command = [sys.executable, str(TV2 / 'run_step3.py'), '--data-root', str(DATA), '--mode', MODE, '--pilot-size', str(PILOT_SIZE)]
print('Đang chạy:', ' '.join(command))
subprocess.run(command, cwd=TV2, check=True)


In [ ]:
import json
from IPython.display import display, Image
REPORTS = DATA / 'tv2_step3' / 'reports'
inventory = json.loads((REPORTS / 'inventory.json').read_text())
print('TV1 NPZ:', inventory['expected'] - len(inventory['missing_tv1']), '/', inventory['expected'])
print('Step 2 VOI:', inventory['available_voi'], '/', inventory['expected'])
print('VOI còn thiếu (tối đa 10 ID đầu):', inventory['missing_voi'][:10])
print(json.dumps(json.loads((REPORTS / 'qc_summary.json').read_text()), indent=2, ensure_ascii=False))
print('Báo cáo và hình đã lưu trên Drive:', REPORTS)
for figure in sorted((REPORTS / 'figures').glob('*.png'))[:4]:
    display(Image(filename=str(figure), width=850))
